# SidSearch LoRA Colab Training
Run these cells top to bottom on a GPU runtime.

In [ ]:
!pip install -q -r requirements-colab.txt
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('memory gb:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
# If starting from a fresh Colab runtime, uncomment and set your repo URL.
# !git clone https://github.com/siddarthasiripragada/llm-lora-distillation-lab.git
# %cd llm-lora-distillation-lab
!python scripts/01_create_protocol.py
!python scripts/02_generate_seed_scenarios.py
!python scripts/05_freeze_benchmark.py
!pytest -q

In [ ]:
from pathlib import Path
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from sidsearch_lora_lab.data.validator import read_jsonl
from sidsearch_lora_lab.training.formatting import format_plain_text
from sidsearch_lora_lab.training.lora import inject_lora, assert_only_lora_trainable
from sidsearch_lora_lab.training.verify_gradients import verify_gradient_flow

model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
model = inject_lora(model)
if torch.cuda.is_available():
    model = model.cuda()
parameter_report = assert_only_lora_trainable(model)
print(json.dumps(parameter_report, indent=2)[:4000])
gradient_report = verify_gradient_flow(model, tokenizer, Path('results/gradient_verification.json'))
print('gradient passed:', gradient_report['passed'])

In [ ]:
train_rows = read_jsonl(Path('data/train.jsonl'))
validation_rows = read_jsonl(Path('data/validation.jsonl'))
def make_dataset(rows):
    texts = [format_plain_text(row) for row in rows]
    ds = Dataset.from_dict({'text': texts})
    def tokenize(batch):
        encoded = tokenizer(batch['text'], truncation=True, max_length=256, padding='max_length')
        encoded['labels'] = [ids[:] for ids in encoded['input_ids']]
        return encoded
    return ds.map(tokenize, batched=True, remove_columns=['text'])
train_ds = make_dataset(train_rows)
eval_ds = make_dataset(validation_rows)
args = TrainingArguments(
    output_dir='adapters/sidsearch-lora',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=[],
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds)
train_output = trainer.train()
trainer.save_model('adapters/sidsearch-lora')
tokenizer.save_pretrained('adapters/sidsearch-lora')
Path('results/train_metrics.json').write_text(json.dumps(train_output.metrics, indent=2), encoding='utf-8')
Path('results/eval_metrics.json').write_text(json.dumps(trainer.evaluate(), indent=2), encoding='utf-8')

In [ ]:
import shutil
archive = shutil.make_archive('sidsearch_lora_adapter_and_metrics', 'zip', '.', 'adapters/sidsearch-lora')
print(archive)
from google.colab import files
files.download(archive)
files.download('results/train_metrics.json')
files.download('results/eval_metrics.json')

In [ ]:
prompt = 'SYSTEM: You are a SidSearch planner. Return exactly one JSON object.\nUSER: Find open repo issues about AtlasSync\nASSISTANT:'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=160, do_sample=False)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))